In [1]:
import os
import re
import pandas as pd
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from peft import PeftModel

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
df = pd.read_csv("../data/slm_eval_dataset.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (20, 2)


,input,expected_risk
0,License: MIT License Family: Permissive Projec...,Low
1,License: MIT License Family: Permissive Projec...,Low
2,License: Apache-2.0 License Family: Permissive...,Low
3,License: BSD-3-Clause License Family: Permissi...,Low
4,License: LGPL-2.1 License Family: Weak Copylef...,Medium


In [3]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(
    base_model,
    "../models/qwen_oss_compliance_lora_v2"
)

print("Fine-tuned Qwen loaded successfully.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Fine-tuned Qwen loaded successfully.


In [4]:
def build_prompt(scenario):
    return f"""
You are an open-source license compliance assistant.

Analyze the following software dependency scenario and classify the compliance risk.

Risk categories:
- Low: permissive licenses with minimal compliance obligations
- Medium: weak copyleft licenses with moderate compliance obligations
- High: strong copyleft, proprietary, unknown, or legally restrictive licenses

Scenario:
{scenario}

Return exactly in this format:

Risk: <Low|Medium|High>
Reason: <one short sentence>
""".strip()

In [5]:
def predict_risk(scenario):
    prompt = build_prompt(scenario)

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [6]:
def extract_risk(response):
    for line in response.split("\n"):
        line = line.strip().lower()

        if line.startswith("risk:"):
            value = line.replace("risk:", "").strip()

            if value.startswith("high"):
                return "High"
            elif value.startswith("medium"):
                return "Medium"
            elif value.startswith("low"):
                return "Low"

    return "Unknown"

In [7]:
notebooks/07_finetuned_qwen_evaluation.ipynb

Processing 1/20
Processing 2/20
Processing 3/20
Processing 4/20
Processing 5/20
Processing 6/20
Processing 7/20
Processing 8/20
Processing 9/20
Processing 10/20
Processing 11/20
Processing 12/20
Processing 13/20
Processing 14/20
Processing 15/20
Processing 16/20
Processing 17/20
Processing 18/20
Processing 19/20
Processing 20/20


,input,expected_risk,finetuned_response,predicted_risk
0,License: MIT License Family: Permissive Projec...,Low,Risk: Low\nReason: Public-domain style terms c...,Low
1,License: MIT License Family: Permissive Projec...,Low,Risk: Low\nReason: Public-domain style terms c...,Low
2,License: Apache-2.0 License Family: Permissive...,Low,Risk: Low\nReason: Permissive terms support re...,Low
3,License: BSD-3-Clause License Family: Permissi...,Low,Risk: Low\nReason: Permissive terms support re...,Low
4,License: LGPL-2.1 License Family: Weak Copylef...,Medium,Risk: Medium\nReason: Moderate compliance requ...,Medium


In [8]:
y_true = df["expected_risk"]
y_pred = df["predicted_risk"]

accuracy = accuracy_score(y_true, y_pred)

report = classification_report(
    y_true,
    y_pred,
    zero_division=0
)

cm = confusion_matrix(y_true, y_pred)

print("Fine-Tuned Qwen Accuracy:", accuracy)

print("\nClassification Report:")
print(report)

print("\nConfusion Matrix:")
print(cm)

Fine-Tuned Qwen Accuracy: 0.8

Classification Report:
              precision    recall  f1-score   support

        High       1.00      1.00      1.00         7
         Low       0.64      1.00      0.78         7
      Medium       1.00      0.33      0.50         6

    accuracy                           0.80        20
   macro avg       0.88      0.78      0.76        20
weighted avg       0.87      0.80      0.77        20


Confusion Matrix:
[[7 0 0]
 [0 7 0]
 [0 4 2]]


In [9]:
comparison_df = pd.DataFrame({
    "Model": [
        "Base Qwen",
        "Fine-Tuned Qwen"
    ],
    "Accuracy": [
        1.0,   # replace if Step 4 changes
        accuracy
    ]
})

comparison_df

,Model,Accuracy
0,Base Qwen,1.0
1,Fine-Tuned Qwen,0.8
